# Demo - Chunking Strategies for RAG
**Day 3 - Session 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day3/demos/notebooks/demo-chunking.ipynb)

**Goal:** Show how fixed-size and overlapping chunking split a document
differently, and why overlap avoids cutting an idea in half at a boundary.

We take one sample document and chunk it three ways: fixed-size with no
overlap, fixed-size with overlap, and paragraph-based. For each we print the
chunk count and a preview so you see the tradeoffs. Overlap repeats a little
text across neighbors so a sentence split by a boundary still appears whole in
one chunk.

No API key, no downloads. Pure Python.

## 1. The sample document
Three paragraphs about QuickBooks. Blank lines separate the paragraphs, which
the paragraph-based strategy will use as boundaries.

In [ ]:
DOCUMENT = """QuickBooks lets small businesses track income and expenses in one place.
You connect a bank account and transactions import automatically each day.
Each transaction can be categorized into a chart of accounts for reporting.

Invoices are created from the Sales menu and can be emailed to customers.
You can set payment terms, add line items, and apply sales tax per region.
Overdue invoices trigger automatic reminders if you enable them in settings.

Reports summarize profit and loss, balance sheet, and cash flow over a period.
You can filter any report by date range, customer, or class before exporting.
Exported reports save as PDF or CSV for sharing with an accountant."""

print(f"Document: {len(DOCUMENT.split())} words, "
      f"{DOCUMENT.count(chr(10) * 2) + 1} paragraphs")

## 2. Chunking helpers
`chunk_fixed` slides a word window with optional overlap, `chunk_paragraphs`
splits on blank lines, and `preview`/`show` print each strategy compactly.

In [ ]:
def chunk_fixed(text, size, overlap=0):
    """Split text into word windows of `size` with `overlap` words repeated."""
    words = text.split()
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunk = words[start:start + size]
        if chunk:
            chunks.append(" ".join(chunk))
        if start + size >= len(words):
            break
    return chunks


def chunk_paragraphs(text):
    """Split on blank lines so each chunk holds one paragraph."""
    return [p.strip().replace("\n", " ") for p in text.split("\n\n") if p.strip()]


def preview(chunk, width=70):
    """First `width` characters of a chunk, for compact printing."""
    flat = " ".join(chunk.split())
    return flat if len(flat) <= width else flat[:width] + "..."


def show(title, chunks):
    """Print a strategy's chunk count and a preview of each chunk."""
    print(f"\n{title}: {len(chunks)} chunks")
    for i, chunk in enumerate(chunks):
        print(f"  [{i}] {preview(chunk)}")

## 3. Compare the three strategies
Same document, three splits. Compare the chunk counts and where boundaries
fall.

In [ ]:
show("Fixed-size (30 words, no overlap)", chunk_fixed(DOCUMENT, 30))
show("Fixed-size (30 words, 8 overlap)", chunk_fixed(DOCUMENT, 30, overlap=8))
show("Paragraph-based", chunk_paragraphs(DOCUMENT))

## Expected output
- Fixed-size with no overlap produces clean windows but can cut a sentence at
  a boundary.
- Fixed-size with overlap repeats a few words across neighbors so a split idea
  still appears whole in one chunk.
- Paragraph-based yields three chunks that respect natural boundaries but vary
  in size.

**Takeaway:** overlap repeats text across chunks so ideas are not cut off;
paragraph splits respect natural boundaries but vary in size.